In [ ]:
import time
import numpy as np
from joblib import Parallel, delayed

def fwht(x):
    """In-place Fast Walsh-Hadamard Transform."""
    x = np.asarray(x, dtype=float).copy()
    L = len(x)
    if L == 0 or (L & (L - 1)) != 0:
        raise ValueError("Length of x must be a power of 2.")
    h = 1
    while h < L:
        y = x.reshape(-1, 2 * h)
        a = y[:, :h].copy()
        b = y[:, h:2*h].copy()
        y[:, :h] = a + b
        y[:, h:2*h] = a - b
        h *= 2
    return x

def gray_permutation(L):
    """Generates the binary-reflected Gray-code permutation indices."""
    i = np.arange(L, dtype=np.int64)
    return i ^ (i >> 1)

def get_ucr_parameters(f_vals):
    """
    Directly computes the UCR parameter vectors beta and gamma
    from the target function values (new mathematical approach).
    """
    eta = np.max(np.abs(f_vals))
    r = np.abs(f_vals) / eta

    # beta_k = 2 * arccos(|f_k| / eta)
    # gamma_k = -2 * arg(f_k)
    beta = 2.0 * np.arccos(r)
    gamma = -2.0 * np.angle(f_vals)

    return beta, gamma

def get_thetas_fast(alpha):
    """O(N log N) Fast Walsh-Hadamard Transform method."""
    L = len(alpha)
    transformed = fwht(alpha)
    gray = gray_permutation(L)
    return transformed[gray] / L

def get_thetas_direct(alpha):
    """O(N^2) Direct Matrix Multiplication method (for validation)."""
    L = len(alpha)
    i = np.arange(L, dtype=np.uint64)
    j = np.arange(L, dtype=np.uint64)
    gray_i = i ^ (i >> np.uint64(1))

    # Compute parity of the bitwise AND
    parity = gray_i[:, None] & j[None, :]
    for shift in [32, 16, 8, 4, 2, 1]:
        parity ^= parity >> np.uint64(shift)
    parity &= np.uint64(1)

    # Map 0 -> 1 and 1 -> -1 to form the Hadamard matrix
    M = 1.0 - 2.0 * parity.astype(float)
    return (M @ alpha) / L

def run_single_trial(m, compute_direct):
    """Runs a single trial for polynomial degree d = 2^m."""
    d = 2**m
    # In the QSP framework, the number of interpolation points is 4d = 2^{m+2}
    N = 4 * d

    # Generate random complex function values
    f_vals = np.random.randn(N) + 1j * np.random.randn(N)

    # 1. Fast Method Timing
    start_fast = time.perf_counter()
    beta, gamma = get_ucr_parameters(f_vals)
    fast_theta_y = get_thetas_fast(beta)
    fast_theta_z = get_thetas_fast(gamma)
    fast_time = time.perf_counter() - start_fast

    direct_time = np.nan
    max_err = np.nan

    # 2. Direct Method Timing & Error Calculation
    if compute_direct:
        start_direct = time.perf_counter()
        direct_theta_y = get_thetas_direct(beta)
        direct_theta_z = get_thetas_direct(gamma)
        direct_time = time.perf_counter() - start_direct

        err_y = np.max(np.abs(fast_theta_y - direct_theta_y))
        err_z = np.max(np.abs(fast_theta_z - direct_theta_z))
        max_err = max(err_y, err_z)

    return fast_time, direct_time, max_err

if __name__ == '__main__':
    # Range of m (Note: vector size N = 2^{m+2})
    # m=24 means N = 2^26 (approx 67 million elements)
    ms = list(range(1, 25))
    num_trials = 10
    max_direct_m = 10       # Cap direct method at m=10 (N=4096) since O(N^2) gets slow

    all_fast_times = []
    all_direct_times = []
    all_errors = []

    print(f"Starting simulations using joblib... ({num_trials} trials per degree)")

    for m in ms:
        compute_direct = (m <= max_direct_m)
        d = 2**m
        print(f"Processing m={m:02d} (d={d:8d}, N={4*d:8d}) | Direct: {compute_direct}")

        # Parallelize the trials for the current m
        results = Parallel(n_jobs=-1)(
            delayed(run_single_trial)(m, compute_direct) for _ in range(num_trials)
        )

        fast_times, direct_times, errors = zip(*results)

        all_fast_times.append(fast_times)
        all_direct_times.append(direct_times)
        all_errors.append(errors)

    # Save data to disk for plotting
    np.savez('simulation_data.npz',
             ms=ms,
             fast_times=all_fast_times,
             direct_times=all_direct_times,
             errors=all_errors)

    print("\n[SUCCESS] Data generated and saved to 'simulation_data.npz'")